# Tree Analysis

In [ ]:
import polars as pl
import xgboost as xgb

from criteo_functions import *
from tree_analysis import *

pl.Config.set_tbl_rows(100)

Cover is the sum of Hessians

Gain is the split improvement score

$$Gain=\frac{1}{2}\left[\frac{G^2_L}{H_L+\lambda} +  \frac{G^2_R}{H_R+\lambda} - \frac{(G_L + G_R)^2}{H_L+H_R + \lambda}\right]-\gamma $$


$$\text{SSE} = \sum_i (y-\hat{y})^2 $$
$$G = 2\sum_{i \in I_j} (\hat{y} - y) $$
$$H = 4\sum_{i \in I_j} 1 $$

for log loss using a logistic function 

$$\sigma(z)=\frac{1}{1+\exp(-z)}$$, 
$$\sigma'(z)=\sigma(z)(1-\sigma(z))$$
$$\sigma''(z)=\sigma'(z)(1-\sigma(z))+ \sigma(z)(1-\sigma'(z))=\sigma(z)(1-\sigma(z))^2+\sigma(z)(1-\sigma(z)(1-\sigma(z)))$$



### Breast Cancer Test Data

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=94)

# Use "hist" for constructing the trees, with early stopping enabled.
clf = xgb.XGBClassifier(
    tree_method="hist", subsample=0.75, gamma=0, reg_lambda=0, early_stopping_rounds=10
)
# Fit the model, test sets are used for early stopping.
clf.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)])
df = get_tree_statistics(clf)
df = add_level(df)

In [ ]:
df.head()

In [ ]:
bst = clf.get_booster()

In [ ]:
df_stats[~df_stats.is_leaf].groupby("depth")["gain"].describe()

In [ ]:
df_stats = tree_node_stats(clf.get_booster())

In [ ]:
df_stats